In [1]:
import pandas as pd

books = pd.read_csv("../data/books_with_categories.csv")

In [2]:
from transformers import pipeline

# classifier for Eckman 6: anger, disgust, fear, joy, neutral, sadness, surprise
classifier = pipeline("text-classification",
                      model="j-hartmann/emotion-english-distilroberta-base",
                      top_k = None,
                      device = "mps")
classifier("I love this!")

config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

[[{'label': 'joy', 'score': 0.9771687984466553},
  {'label': 'surprise', 'score': 0.008528691716492176},
  {'label': 'neutral', 'score': 0.005764603149145842},
  {'label': 'anger', 'score': 0.004419785924255848},
  {'label': 'sadness', 'score': 0.0020923931151628494},
  {'label': 'disgust', 'score': 0.0016119939973577857},
  {'label': 'fear', 'score': 0.0004138521908316761}]]

In [7]:
books["description"][0]

'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the best and the worst the world ha

In [ ]:
classifier(books["description"][0])
# not a very accurate classification - fear should not be so heavily favored for this book

[[{'label': 'fear', 'score': 0.6548412442207336},
  {'label': 'neutral', 'score': 0.16985206305980682},
  {'label': 'sadness', 'score': 0.11640891432762146},
  {'label': 'surprise', 'score': 0.020700661465525627},
  {'label': 'disgust', 'score': 0.019100770354270935},
  {'label': 'joy', 'score': 0.015161288902163506},
  {'label': 'anger', 'score': 0.003935153596103191}]]

In [ ]:
# classify sentences individually
classifier(books["description"][0].split("."))

[[{'label': 'surprise', 'score': 0.7296032905578613},
  {'label': 'neutral', 'score': 0.14038535952568054},
  {'label': 'fear', 'score': 0.06816209107637405},
  {'label': 'joy', 'score': 0.047942325472831726},
  {'label': 'anger', 'score': 0.009156342595815659},
  {'label': 'disgust', 'score': 0.002628472400829196},
  {'label': 'sadness', 'score': 0.0021221591159701347}],
 [{'label': 'neutral', 'score': 0.4493713974952698},
  {'label': 'disgust', 'score': 0.27359098196029663},
  {'label': 'joy', 'score': 0.10908278077840805},
  {'label': 'sadness', 'score': 0.09362737089395523},
  {'label': 'anger', 'score': 0.04047822952270508},
  {'label': 'surprise', 'score': 0.026970190927386284},
  {'label': 'fear', 'score': 0.006879057735204697}],
 [{'label': 'neutral', 'score': 0.6462154984474182},
  {'label': 'sadness', 'score': 0.24273385107517242},
  {'label': 'disgust', 'score': 0.04342261701822281},
  {'label': 'surprise', 'score': 0.028300536796450615},
  {'label': 'joy', 'score': 0.014211

In [ ]:
sentences = books["description"][0].split(".")
predictions = classifier(sentences)
sorted(predictions[0], key=lambda x: x["label"])
# take the sentence with the highest probability for each sentiment

[{'label': 'anger', 'score': 0.009156342595815659},
 {'label': 'disgust', 'score': 0.002628472400829196},
 {'label': 'fear', 'score': 0.06816209107637405},
 {'label': 'joy', 'score': 0.047942325472831726},
 {'label': 'neutral', 'score': 0.14038535952568054},
 {'label': 'sadness', 'score': 0.0021221591159701347},
 {'label': 'surprise', 'score': 0.7296032905578613}]

In [ ]:
import numpy as np

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

# creates a dictionary for each description containing the maximumm probability for each emotion
def calculate_max_emotion_scores(predictions):
    per_emotion_scores = {label: [] for label in emotion_labels}
    for prediction in predictions:
        sorted_predictions = sorted(prediction, key=lambda x: x["label"])
        for index, label in enumerate(emotion_labels):
            per_emotion_scores[label].append(sorted_predictions[index]["score"])
    return {label: np.max(scores) for label, scores in per_emotion_scores.items()}

In [ ]:
# test for the first 10 books
for i in range(10):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

In [13]:
emotion_scores

{'anger': [np.float64(0.06413351744413376),
  np.float64(0.6126188039779663),
  np.float64(0.06413351744413376),
  np.float64(0.351483553647995),
  np.float64(0.0814124271273613),
  np.float64(0.23222501575946808),
  np.float64(0.5381842255592346),
  np.float64(0.06413351744413376),
  np.float64(0.3006700873374939),
  np.float64(0.06413351744413376),
  np.float64(0.06413351744413376),
  np.float64(0.6126188039779663),
  np.float64(0.06413351744413376),
  np.float64(0.351483553647995),
  np.float64(0.0814124271273613),
  np.float64(0.23222501575946808),
  np.float64(0.5381842255592346),
  np.float64(0.06413351744413376),
  np.float64(0.3006700873374939),
  np.float64(0.06413351744413376)],
 'disgust': [np.float64(0.27359098196029663),
  np.float64(0.34828582406044006),
  np.float64(0.1040065661072731),
  np.float64(0.1507226824760437),
  np.float64(0.1844950169324875),
  np.float64(0.727174699306488),
  np.float64(0.15585489571094513),
  np.float64(0.1040065661072731),
  np.float64(0.27

In [14]:
from tqdm import tqdm

emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]
isbn = []
emotion_scores = {label: [] for label in emotion_labels}

for i in tqdm(range(len(books))):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

100%|██████████| 5693/5693 [18:02<00:00,  5.26it/s]


In [15]:
emotions_df = pd.DataFrame(emotion_scores)
emotions_df["isbn13"] = isbn

In [16]:
emotions_df

,anger,disgust,fear,joy,sadness,surprise,neutral,isbn13
0,0.064134,0.273591,0.928169,0.932798,0.646215,0.967157,0.729603,9780002005883
1,0.612619,0.348286,0.942528,0.704421,0.887939,0.111690,0.252545,9780002261982
2,0.064134,0.104007,0.972321,0.767237,0.549477,0.111690,0.078765,9780006178736
3,0.351484,0.150723,0.360707,0.251881,0.732686,0.111690,0.078765,9780006280897
4,0.081412,0.184495,0.095043,0.040564,0.884390,0.475881,0.078765,9780006280934
...,...,...,...,...,...,...,...,...
5688,0.064134,0.114383,0.051363,0.400263,0.883198,0.111690,0.227765,9788173031014
5689,0.009997,0.009929,0.339217,0.947779,0.375756,0.066685,0.057625,9788179921623
5690,0.064134,0.104007,0.459268,0.759455,0.951104,0.368110,0.078765,9788185300535
5691,0.064134,0.104007,0.051363,0.958549,0.915193,0.111690,0.078765,9789027712059


In [17]:
# add the maximum score for each emotion to the book vectors
books = pd.merge(books, emotions_df, on = "isbn13")

In [18]:
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,...,title_and_subtitle,tagged_description,simple_categories,anger,disgust,fear,joy,sadness,surprise,neutral
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,...,Gilead,9780002005883 A NOVEL THAT READERS and critics...,Fiction,0.064134,0.273591,0.928169,0.932798,0.646215,0.967157,0.729603
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,...,Spider's Web: A Novel,9780002261982 A new 'Christie for Christmas' -...,Fiction,0.612619,0.348286,0.942528,0.704421,0.887939,0.111690,0.252545
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,...,Rage of angels,"9780006178736 A memorable, mesmerizing heroine...",Fiction,0.064134,0.104007,0.972321,0.767237,0.549477,0.111690,0.078765
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,...,The Four Loves,9780006280897 Lewis' work on the nature of lov...,Nonfiction,0.351484,0.150723,0.360707,0.251881,0.732686,0.111690,0.078765
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,...,The Problem of Pain,"9780006280934 ""In The Problem of Pain, C.S. Le...",Nonfiction,0.081412,0.184495,0.095043,0.040564,0.884390,0.475881,0.078765
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5688,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,...,Journey to the East,9788173031014 This book tells the tale of a ma...,Nonfiction,0.064134,0.114383,0.051363,0.400263,0.883198,0.111690,0.227765
5689,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,...,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623 Wisdom to Create a Life of Passi...,Fiction,0.009997,0.009929,0.339217,0.947779,0.375756,0.066685,0.057625
5690,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,...,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535 This collection of the timeless ...,Nonfiction,0.064134,0.104007,0.459268,0.759455,0.951104,0.368110,0.078765
5691,9789027712059,9027712050,The Berlin Phenomenology,Georg Wilhelm Friedrich Hegel,History,http://books.google.com/books/content?id=Vy7Sk...,Since the three volume edition ofHegel's Philo...,1981.0,0.00,210.0,...,The Berlin Phenomenology,9789027712059 Since the three volume edition o...,Nonfiction,0.064134,0.104007,0.051363,0.958549,0.915193,0.111690,0.078765


In [19]:
books.to_csv("../data/books_with_emotions.csv", index = False)